# Lesson 2: Data Preparation and ML Workflow

## Python Machine Learning for High School Students

In this notebook, you will practice the basic machine learning workflow:

**raw data → explore data → clean data → prepare features → split data → train model → evaluate model → improve model**

We will use a small student-performance dataset. The goal is to predict whether a student is likely to pass based on study habits and school-related information.

> **Important:** This is a classroom learning example, not a real system for judging students. Real education data can be sensitive and should be handled carefully.


# Learning Objectives

By the end of this notebook, you will be able to:

1. Load data into a pandas DataFrame.
2. Explore a dataset using `.head()`, `.info()`, `.describe()`, and missing-value checks.
3. Separate features `X` from the target label `y`.
4. Handle missing numerical values.
5. Handle missing categorical values.
6. Convert categorical text data into numbers.
7. Split data into training and test sets.
8. Train a simple machine learning model.
9. Evaluate the model using accuracy and a confusion matrix.
10. Improve the workflow using a second model.


# Part 1: Background

Machine learning models learn patterns from data.

For supervised learning, we usually have:

- **Features `X`**: the input information used to make a prediction.
- **Target `y`**: the answer we want the model to predict.

In this notebook:

- Features include `hours_studied`, `attendance_rate`, `previous_score`, `internet_access`, and `parent_support`.
- Target is `passed`.

The target uses:

- `0` = did not pass
- `1` = passed

Before training a model, we must prepare the data. Many real datasets contain missing values, text labels, or messy columns. A model usually needs clean numerical data.


## Teacher Note

This notebook intentionally uses a small artificial dataset so students can understand the workflow clearly. In a real STEM competition project, students should use a larger dataset and spend more time on data quality, ethics, and model validation.


# Part 2: Load the Dataset

First, we import the Python libraries and create a small dataset.

We will use:

- `pandas` for working with tables
- `numpy` for missing values and numerical operations
- `matplotlib` for simple visualizations
- `scikit-learn` for preprocessing, model training, and evaluation


In [ ]:
# Import libraries

# pandas helps us work with table-like data.
import pandas as pd

# numpy helps us work with numbers and missing values.
import numpy as np

# matplotlib helps us create simple charts.
import matplotlib.pyplot as plt

# train_test_split divides data into training and test sets.
from sklearn.model_selection import train_test_split

# SimpleImputer fills missing values.
from sklearn.impute import SimpleImputer

# OneHotEncoder converts text categories into numbers.
from sklearn.preprocessing import OneHotEncoder

# ColumnTransformer applies different preprocessing steps to different columns.
from sklearn.compose import ColumnTransformer

# Pipeline connects preprocessing and model training into one workflow.
from sklearn.pipeline import Pipeline

# LogisticRegression is a simple classification model.
from sklearn.linear_model import LogisticRegression

# RandomForestClassifier is a more flexible classification model.
from sklearn.ensemble import RandomForestClassifier

# These tools help us evaluate the model.
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


## Create the Dataset

This dataset is small and easy to understand. Some values are intentionally missing so we can practice data preparation.


In [ ]:
# Create a small artificial dataset about student study habits.

data = {
    "hours_studied": [2, 5, 1, 8, 6, np.nan, 3, 7, 4, 9, 2, 6, np.nan, 5, 10],
    "attendance_rate": [60, 85, 50, 95, 90, 70, np.nan, 88, 75, 98, 55, 82, 65, np.nan, 99],
    "previous_score": [55, 78, 40, 88, 84, 62, 58, 80, np.nan, 92, 50, 76, 60, 74, 96],
    "internet_access": ["No", "Yes", "No", "Yes", "Yes", "No", "Yes", np.nan, "Yes", "Yes", "No", "Yes", "No", "Yes", "Yes"],
    "parent_support": ["Low", "Medium", "Low", "High", "High", "Medium", np.nan, "High", "Medium", "High", "Low", "Medium", "Low", "Medium", "High"],
    "passed": [0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1]
}

# Convert the dictionary into a pandas DataFrame.
df = pd.DataFrame(data)

# Display the full dataset.
df


# Part 3: Explore the Data

Before building a model, we should understand the data.

In this section, we will:

1. Preview the first rows.
2. Check the shape of the dataset.
3. Check column types.
4. Find missing values.
5. Create a simple chart.


## Student TODO 1: Preview the Dataset

Complete the code below.


In [ ]:
# TODO: Display the first 5 rows of the dataset.

# Hint: Use df.head()



## Teacher Solution 1

In [ ]:
# Teacher solution: Display the first 5 rows.
df.head()


## Student TODO 2: Check Dataset Size and Column Information

In [ ]:
# TODO: Print the number of rows and columns.

# Hint: Use df.shape



In [ ]:
# TODO: Display column names, data types, and missing-value information.

# Hint: Use df.info()



## Teacher Solution 2

In [ ]:
# Teacher solution: Print the number of rows and columns.
print("Dataset shape:")
print(df.shape)

# Teacher solution: Display column information.
print("\nDataset information:")
df.info()


## Student TODO 3: Check Missing Values

In [ ]:
# TODO: Count missing values in each column.

# Hint: Use df.isnull().sum()



## Teacher Solution 3

In [ ]:
# Teacher solution: Count missing values in each column.
df.isnull().sum()


## Visual Exploration

Let’s create a simple chart to see the relationship between study hours and previous score.

This chart does not train a model. It helps us understand the data.


In [ ]:
# Create a scatter plot.

# The x-axis shows hours studied.
plt.scatter(df["hours_studied"], df["previous_score"])

# Add axis labels and a title.
plt.xlabel("Hours Studied")
plt.ylabel("Previous Score")
plt.title("Hours Studied vs. Previous Score")

# Show the chart.
plt.show()


# Part 4: Prepare the Data

Machine learning models need data in the right format.

We need to:

1. Separate features `X` and target `y`.
2. Identify numerical and categorical columns.
3. Fill missing numerical values.
4. Fill missing categorical values.
5. Convert categorical values into numbers.


## Student TODO 4: Separate Features and Target

The target column is `passed`.

Features are all columns except `passed`.


In [ ]:
# TODO: Create X by dropping the passed column.
# TODO: Create y using the passed column.

# X = ...
# y = ...



## Teacher Solution 4

In [ ]:
# Teacher solution: Separate input features and target label.

# X contains the input columns used to make predictions.
X = df.drop("passed", axis=1)

# y contains the answer the model should learn to predict.
y = df["passed"]

# Display the first few rows of X.
print("Features X:")
display(X.head())

# Display the first few values of y.
print("Target y:")
display(y.head())


## Identify Column Types

We will treat these columns differently:

- Numerical columns: fill missing values using the median.
- Categorical columns: fill missing values using the most frequent value, then one-hot encode them.


In [ ]:
# List of numerical feature columns.
numerical_features = ["hours_studied", "attendance_rate", "previous_score"]

# List of categorical feature columns.
categorical_features = ["internet_access", "parent_support"]


## Student TODO 5: Create Preprocessing Steps

Complete the preprocessing code.

Hints:

- For numerical data, use `SimpleImputer(strategy="median")`.
- For categorical data, use `SimpleImputer(strategy="most_frequent")`.
- Then use `OneHotEncoder(handle_unknown="ignore")`.


In [ ]:
# TODO: Create a numerical transformer that fills missing numbers with the median.

# numerical_transformer = ...


# TODO: Create a categorical transformer pipeline.
# It should first fill missing categories with the most frequent value.
# Then it should one-hot encode the categorical values.

# categorical_transformer = Pipeline(steps=[
#     ("imputer", ...),
#     ("onehot", ...)
# ])


# TODO: Combine the numerical and categorical transformers.

# preprocessor = ColumnTransformer(
#     transformers=[
#         ("num", numerical_transformer, numerical_features),
#         ("cat", categorical_transformer, categorical_features)
#     ]
# )



## Teacher Solution 5

In [ ]:
# Teacher solution: Create a numerical transformer.

# This fills missing numerical values with the median.
numerical_transformer = SimpleImputer(strategy="median")

# Teacher solution: Create a categorical transformer.

# This pipeline first fills missing categorical values,
# then converts text categories into numbers.
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Teacher solution: Combine preprocessing steps.

# ColumnTransformer allows us to apply different preprocessing
# to different groups of columns.
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Preprocessor created successfully.")


# Part 5: Train the Model

Now we will train a simple classification model.

We will use **Logistic Regression** because our target is binary:

- `0` = did not pass
- `1` = passed

We will put preprocessing and model training into one `Pipeline`.


## Student TODO 6: Create a Pipeline

In [ ]:
# TODO: Create a pipeline with two steps:
# 1. preprocessor
# 2. LogisticRegression

# model = Pipeline(steps=[
#     ("preprocessor", ...),
#     ("classifier", ...)
# ])



## Teacher Solution 6

In [ ]:
# Teacher solution: Create the full machine learning pipeline.

# The pipeline first prepares the data.
# Then it trains the classification model.
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression())
])

print("Model pipeline created successfully.")


## Student TODO 7: Split the Data

We split the data into:

- Training set: used to teach the model
- Test set: used to check the model on new examples

Use:

- `test_size=0.3`
- `random_state=42`


In [ ]:
# TODO: Split X and y into training and test sets.

# X_train, X_test, y_train, y_test = train_test_split(
#     X,
#     y,
#     test_size=...,
#     random_state=...
# )



## Teacher Solution 7

In [ ]:
# Teacher solution: Split data into training and test sets.

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42
)

# Print the sizes of the training and test sets.
print("Training features shape:", X_train.shape)
print("Test features shape:", X_test.shape)
print("Training target shape:", y_train.shape)
print("Test target shape:", y_test.shape)


## Student TODO 8: Train the Model

In [ ]:
# TODO: Train the model using X_train and y_train.

# model.fit(..., ...)



## Teacher Solution 8

In [ ]:
# Teacher solution: Train the model.

model.fit(X_train, y_train)

print("Model training complete.")


# Part 6: Evaluate the Model

After training, we need to check how well the model performs.

We will use:

1. **Accuracy**: percentage of correct predictions.
2. **Confusion matrix**: shows correct and incorrect predictions.
3. **Classification report**: shows precision, recall, and F1-score.


## Student TODO 9: Make Predictions

In [ ]:
# TODO: Use the trained model to predict labels for X_test.

# y_pred = ...



## Teacher Solution 9

In [ ]:
# Teacher solution: Make predictions on the test set.

y_pred = model.predict(X_test)

print("Predicted labels:")
print(y_pred)

print("\nActual labels:")
print(y_test.values)


## Student TODO 10: Calculate Accuracy

In [ ]:
# TODO: Calculate the model accuracy.

# accuracy = ...
# print("Accuracy:", accuracy)



## Teacher Solution 10

In [ ]:
# Teacher solution: Calculate accuracy.

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)


## Confusion Matrix and Classification Report

In [ ]:
# Print the confusion matrix.

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


## How to Read the Confusion Matrix

For this binary classification problem:

- `0` means did not pass.
- `1` means passed.

The confusion matrix shows:

```text
[[actual 0 predicted 0, actual 0 predicted 1],
 [actual 1 predicted 0, actual 1 predicted 1]]
```

In simple words, it tells us where the model was correct and where it made mistakes.


# Part 7: Improve the Model

A good machine learning workflow often compares more than one model.

Now we will try a **Random Forest Classifier**.

Random Forest is often stronger than a single simple model because it combines many decision trees.


## Student TODO 11: Train a Random Forest Model

In [ ]:
# TODO: Create a new pipeline using RandomForestClassifier.
# Use random_state=42.

# rf_model = Pipeline(steps=[
#     ("preprocessor", ...),
#     ("classifier", ...)
# ])

# TODO: Train the random forest model.
# rf_model.fit(..., ...)

# TODO: Make predictions.
# rf_pred = ...

# TODO: Calculate accuracy.
# rf_accuracy = ...

# print("Random Forest Accuracy:", rf_accuracy)



## Teacher Solution 11

In [ ]:
# Teacher solution: Create a Random Forest pipeline.

rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])

# Train the Random Forest model.
rf_model.fit(X_train, y_train)

# Make predictions using the Random Forest model.
rf_pred = rf_model.predict(X_test)

# Calculate Random Forest accuracy.
rf_accuracy = accuracy_score(y_test, rf_pred)

print("Logistic Regression Accuracy:", accuracy)
print("Random Forest Accuracy:", rf_accuracy)

print("\nRandom Forest Confusion Matrix:")
print(confusion_matrix(y_test, rf_pred))


## Important Discussion

Because this dataset is very small, a high accuracy score does **not** prove that the model is truly reliable.

In real projects, we need:

- More rows of data
- Better features
- Careful train/test splitting
- Cross-validation
- Ethical review
- Error analysis
- Clear explanation of limitations


# Part 8: Reflection Questions

Answer these questions in your notebook or on a worksheet.

## Data Understanding

1. What is the target variable in this dataset?
2. Which columns are numerical features?
3. Which columns are categorical features?
4. Which columns had missing values?

## Data Preparation

5. Why did we fill missing numerical values with the median?
6. Why did we fill missing categorical values with the most frequent value?
7. Why do we need one-hot encoding?
8. Why do we split the dataset into training and test sets?

## Model Evaluation

9. What does accuracy measure?
10. What does the confusion matrix show?
11. Did Logistic Regression and Random Forest give the same result?
12. Why should we be careful about trusting results from a very small dataset?

## STEM Competition Thinking

13. What would be a good research question based on this dataset?
14. What are two ethical concerns with using student data?
15. What additional feature might improve the model?


# Part 9: Challenge Task

Choose one or more challenges.

## Challenge A: Add a New Feature

Add a new column called `sleep_hours`.

Suggested values:

```python
[6, 8, 5, 7, 8, 6, 5, 7, 6, 8, 5, 7, 6, 7, 8]
```

Then retrain the model.

Questions:

1. Did the model accuracy change?
2. Why might sleep be related to student performance?
3. Would sleep data be private or sensitive?

---

## Challenge B: Change the Test Size

Try:

```python
test_size=0.2
test_size=0.4
```

Questions:

1. Did the accuracy change?
2. Why can the train/test split affect results?
3. What problem happens when the dataset is too small?

---

## Challenge C: Create a STEM Competition Project Idea

Write a short project idea using this format:

```text
Project Title:
Problem:
Research Question:
Dataset Needed:
Features:
Target:
Model:
Evaluation Metric:
Ethical Concerns:
Expected Impact:
```

Example:

```text
Project Title: Predicting Student Learning Risk from Study Habits

Problem: Some students fall behind before teachers notice.

Research Question: Can study habits and attendance patterns help identify students who may need extra support?

Dataset Needed: Anonymous student study and attendance records.

Features: Study hours, attendance, previous quiz scores, homework completion.

Target: Whether the student needs academic support.

Model: Logistic Regression or Random Forest.

Evaluation Metric: Accuracy, precision, recall, and confusion matrix.

Ethical Concerns: Privacy, bias, fairness, and avoiding harmful labels.

Expected Impact: Help teachers provide earlier support to students.
```


# Teacher Solution for Challenge A

Run this only after students try the challenge.


In [ ]:
# Teacher solution for Challenge A: Add sleep_hours and retrain the model.

# Make a copy of the original dataset so we do not accidentally damage it.
df_challenge = df.copy()

# Add a new feature.
df_challenge["sleep_hours"] = [6, 8, 5, 7, 8, 6, 5, 7, 6, 8, 5, 7, 6, 7, 8]

# Create new X and y.
X_challenge = df_challenge.drop("passed", axis=1)
y_challenge = df_challenge["passed"]

# Update the numerical features list to include sleep_hours.
numerical_features_challenge = ["hours_studied", "attendance_rate", "previous_score", "sleep_hours"]

# Keep the same categorical features.
categorical_features_challenge = ["internet_access", "parent_support"]

# Create a new preprocessor for the challenge dataset.
preprocessor_challenge = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features_challenge),
        ("cat", categorical_transformer, categorical_features_challenge)
    ]
)

# Split the challenge dataset.
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_challenge,
    y_challenge,
    test_size=0.3,
    random_state=42
)

# Create a challenge model pipeline.
challenge_model = Pipeline(steps=[
    ("preprocessor", preprocessor_challenge),
    ("classifier", LogisticRegression())
])

# Train the challenge model.
challenge_model.fit(X_train_c, y_train_c)

# Make predictions.
challenge_pred = challenge_model.predict(X_test_c)

# Evaluate the challenge model.
challenge_accuracy = accuracy_score(y_test_c, challenge_pred)

print("Original Logistic Regression Accuracy:", accuracy)
print("Challenge Model Accuracy:", challenge_accuracy)


# Final Summary

In this notebook, you practiced the core machine learning workflow:

1. Load data.
2. Explore data.
3. Identify missing values.
4. Separate features and target.
5. Preprocess numerical and categorical features.
6. Split data into training and test sets.
7. Train a model.
8. Evaluate the model.
9. Compare models.
10. Think about ethics and STEM competition project design.

This workflow will appear again and again in future lessons.
